Currently planned to compare 5 different models: 
- Dummy baseline
- Logistic Regression
- Regularised Logistic Regression
- Random Forest
- XGBoost

## Data Preperation

In [45]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


In [20]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv")

df = pd.read_csv(DATA_PATH)

print(DATA_PATH)
print(df.shape)
df.head()

/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


In [21]:
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted',
 'readmitted_30',
 'hba1c_group',
 'primary_diagnosis',
 'age_group',
 'discharge_group',
 'race_group',
 'admission_source_group',
 'medical_specialty_group']

In [3]:
df["readmitted"].value_counts(dropna=False)

readmitted
NO     41476
>30    22226
<30     6285
Name: count, dtype: int64

In [4]:
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

df["readmitted_30"].value_counts(normalize=True)

readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64

In [5]:
target_col = "readmitted_30"

drop_cols = [
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
]

# Drop columns only if they actually exist
drop_cols_existing = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols_existing)
y = df[target_col]

print(X.shape)
print(y.value_counts())
print(y.value_counts(normalize=True))

(69987, 52)
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, y_train.mean())
print("Test:", X_test.shape, y_test.mean())

Train: (55989, 52) 0.08980335423029524
Test: (13998, 52) 0.08979854264894985


In [9]:
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

print("Categorical features:", len(categorical_features))
print(categorical_features)

print("Numeric features:", len(numeric_features))
print(numeric_features)

Categorical features: 41
['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'hba1c_group', 'primary_diagnosis', 'age_group', 'discharge_group', 'race_group', 'admission_source_group', 'medical_specialty_group']
Numeric features: 11
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']


In [10]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features)
    ]
)

In [32]:
def save_confusion_matrix(model, X_test, y_test, model_name, output_dir):
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual not readmitted", "Actual readmitted"],
        columns=["Predicted not readmitted", "Predicted readmitted"]
    )

    file_name = (
        model_name
        .lower()
        .replace(":", "")
        .replace(" ", "_")
        .replace("/", "_")
    )

    cm_df.to_csv(output_dir / f"confusion_matrix_{file_name}.csv")

    tn, fp, fn, tp = cm.ravel()

    total = tn + fp + fn + tp

    cm_long = pd.DataFrame([{
        "model": model_name,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "total": total,
        "true_negative_rate": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "false_positive_rate": fp / (tn + fp) if (tn + fp) > 0 else 0,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "true_positive_rate_recall": tp / (fn + tp) if (fn + tp) > 0 else 0
    }])

    return cm_df, cm_long

## Dummy Model

In [11]:
dummy_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](52,)","['race','gender','age',...,'race_group','admission_source_group', 'medical_specialty_group']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,52
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [12]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    
    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_test, y_proba),
        "auprc": average_precision_score(y_test, y_proba),
        "brier_score": brier_score_loss(y_test, y_proba)
    }
    
    return results

In [13]:
dummy_results = evaluate_model(
    dummy_model,
    X_test,
    y_test,
    model_name="Dummy: most frequent"
)

dummy_results

{'model': 'Dummy: most frequent',
 'accuracy': 0.9102014573510502,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'auroc': 0.5,
 'auprc': 0.08979854264894985,
 'brier_score': 0.08979854264894985}

In [14]:
results_df = pd.DataFrame([dummy_results])
results_df

,model,accuracy,precision,recall,f1,auroc,auprc,brier_score
0,Dummy: most frequent,0.910201,0.0,0.0,0.0,0.5,0.089799,0.089799


In [38]:
OUTPUT_DIR = PROJECT_ROOT / "Model_Results"
OUTPUT_DIR.mkdir(exist_ok=True)

results_df.to_csv(OUTPUT_DIR / "model_comparison_initial.csv", index=False)
dummy_cm_df, dummy_cm_long = save_confusion_matrix(
    model=dummy_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Dummy: most frequent",
    output_dir=OUTPUT_DIR
)

In [39]:
dummy_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,12741,0
Actual readmitted,1257,0


In [40]:
dummy_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Dummy: most frequent,12741,0,1257,0,13998,1.0,0.0,1.0,0.0


In [18]:
extra_drop_cols = [
    col for col in df.columns
    if "table" in col.lower() or "display" in col.lower()
]

extra_drop_cols

[]

In [44]:
y_dummy_pred = dummy_model.predict(X_test)

print(classification_report(
    y_test,
    y_dummy_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.91      1.00      0.95     12741
    Readmitted       0.00      0.00      0.00      1257

      accuracy                           0.91     13998
     macro avg       0.46      0.50      0.48     13998
  weighted avg       0.83      0.91      0.87     13998



## Logistic Regression Model

In [22]:
log_reg_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](52,)","['race','gender','age',...,'race_group','admission_source_group', 'medical_specialty_group']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,52
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [23]:
log_reg_results = evaluate_model(
    log_reg_model,
    X_test,
    y_test,
    model_name="Logistic Regression"
)

log_reg_results

{'model': 'Logistic Regression',
 'accuracy': 0.6365195027861124,
 'precision': 0.12769679300291545,
 'recall': 0.522673031026253,
 'f1': 0.20524835988753515,
 'auroc': 0.6168224444952706,
 'auprc': 0.14610525508957029,
 'brier_score': 0.22392973465870175}

In [42]:
results_df.to_csv(
    OUTPUT_DIR / "model_comparison_initial.csv",
    index=False
)

log_reg_cm_df, log_reg_cm_long = save_confusion_matrix(
    model=log_reg_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Logistic Regression",
    output_dir=OUTPUT_DIR
)

In [43]:
log_reg_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,8253,4488
Actual readmitted,600,657


In [41]:
log_reg_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Logistic Regression,8253,4488,600,657,13998,0.647751,0.352249,0.477327,0.522673


In [35]:
y_logreg_pred = log_reg_model.predict(X_test)

print(classification_report(
    y_test,
    y_logreg_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.93      0.65      0.76     12741
    Readmitted       0.13      0.52      0.21      1257

      accuracy                           0.64     13998
     macro avg       0.53      0.59      0.48     13998
  weighted avg       0.86      0.64      0.71     13998



## Regularised Logistic Regression 

In [46]:
regularised_log_reg_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=5000,
        tol=1e-3,
        random_state=42
    ))
])

regularised_log_reg_param_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.01, 0.1, 1, 10]
}

regularised_log_reg_search = GridSearchCV(
    estimator=regularised_log_reg_pipeline,
    param_grid=regularised_log_reg_param_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=1
)

regularised_log_reg_search.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/opt/anaconda3/envs/indi_preprocessing/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/indi_preprocessing/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/anaconda3/envs/indi_preprocessing/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l1', 'l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`

In [47]:
regularised_log_reg_search.best_params_

{'model__C': 0.1, 'model__penalty': 'l1'}

In [48]:
regularised_log_reg_search.best_score_

np.float64(0.15389032631288407)

In [50]:
regularised_log_reg_model = regularised_log_reg_search.best_estimator_

reg_log_reg_results = evaluate_model(
    regularised_log_reg_model,
    X_test,
    y_test,
    model_name="Regularised Logistic Regression"
)

reg_log_reg_results

{'model': 'Regularised Logistic Regression',
 'accuracy': 0.6295899414202029,
 'precision': 0.13275991024682124,
 'recall': 0.5648369132856006,
 'f1': 0.2149886449659349,
 'auroc': 0.6391058202158331,
 'auprc': 0.1601328677419055,
 'brier_score': 0.23013051637113005}

In [51]:
# Rebuild the comparison table safely.
# This avoids duplicate rows if you rerun the notebook cells.

available_results = []

if "dummy_results" in globals():
    available_results.append(dummy_results)

if "log_reg_results" in globals():
    available_results.append(log_reg_results)

if "reg_log_reg_results" in globals():
    available_results.append(reg_log_reg_results)

results_df = pd.DataFrame(available_results)

results_df.to_csv(
    OUTPUT_DIR / "model_comparison_initial.csv",
    index=False
)

results_df

,model,accuracy,precision,recall,f1,auroc,auprc,brier_score
0,Dummy: most frequent,0.910201,0.000000,0.000000,0.000000,0.500000,0.089799,0.089799
1,Logistic Regression,0.636520,0.127697,0.522673,0.205248,0.616822,0.146105,0.223930
2,Regularised Logistic Regression,0.629590,0.132760,0.564837,0.214989,0.639106,0.160133,0.230131


In [52]:
reg_log_reg_cm_df, reg_log_reg_cm_long = save_confusion_matrix(
    model=regularised_log_reg_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Regularised Logistic Regression",
    output_dir=OUTPUT_DIR
)

In [53]:
reg_log_reg_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,8103,4638
Actual readmitted,547,710


In [54]:
reg_log_reg_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Regularised Logistic Regression,8103,4638,547,710,13998,0.635978,0.364022,0.435163,0.564837


In [55]:
y_reg_logreg_pred = regularised_log_reg_model.predict(X_test)

print(classification_report(
    y_test,
    y_reg_logreg_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.94      0.64      0.76     12741
    Readmitted       0.13      0.56      0.21      1257

      accuracy                           0.63     13998
     macro avg       0.53      0.60      0.49     13998
  weighted avg       0.86      0.63      0.71     13998



In [56]:
regularised_log_reg_best_params = pd.DataFrame([{
    "model": "Regularised Logistic Regression",
    "best_penalty": regularised_log_reg_search.best_params_["model__penalty"],
    "best_C": regularised_log_reg_search.best_params_["model__C"],
    "best_cv_auprc": regularised_log_reg_search.best_score_
}])

regularised_log_reg_best_params.to_csv(
    OUTPUT_DIR / "regularised_logistic_regression_best_params.csv",
    index=False
)

regularised_log_reg_best_params

,model,best_penalty,best_C,best_cv_auprc
0,Regularised Logistic Regression,l1,0.1,0.15389
